In [ ]:
import os
import numpy as np
from dotenv import load_dotenv
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.wcs import WCS
from astropy.io import fits
load_dotenv("/home/yacheng/nexus/ssl_outthere/.env")

if not os.path.exists("spectra-fitting.fits"):
    !wget --user="$OUTTHERE_USER" --password="$OUTTHERE_PASSWORD" \
        https://outthere-mpia.org/s3/data/spectra-fitting.fits

if not os.path.exists("phomoetry.fits"):
    !wget --user="$OUTTHERE_USER" --password="$OUTTHERE_PASSWORD" \
        https://outthere-mpia.org/s3/data/phomoetry.fits



In [ ]:
fields = [
    "aqr-00", "aqr-01", "aqr-02",
    "boo-00", "boo-01", "boo-02", "boo-03", "boo-04", "boo-05", "boo-06",
    "boo-07", "boo-08", "boo-09", "boo-10", "boo-11",
    "cam-00",
    "cet-00", "cet-01", "cet-02", "cet-03", "cet-04", "cet-05", "cet-06",
    "cet-07", "cet-08", "cet-09", "cet-10",
    "col-00",
    "com-00", "com-01", "com-02",
    "crb-00",
    "crt-00", "crt-01",
    "crv-00",
    "cvn-00",
    "dor-00", "dor-01", "dor-02", "dor-03",
    "dra-00", "dra-01", "dra-02",
    "eri-00",
    "gru-00",
    "her-00", "her-01",
    "hya-00", "hya-01", "hya-02",
    "leo-00", "leo-01", "leo-02", "leo-03", "leo-04", "leo-05", "leo-06",
    "leo-07", "leo-08", "leo-09", "leo-10", "leo-11", "leo-12", "leo-13",
    "leo-14", "leo-15", "leo-16", "leo-17",
    "lib-00", "lib-01", "lib-02", "lib-03",
    "lmi-00", "lmi-01",
    "lyn-00",
    "psc-00",
    "sex-00", "sex-01", "sex-02", "sex-03", "sex-04", "sex-05", "sex-06",
    "sex-07", "sex-08", "sex-09", "sex-10", "sex-11", "sex-12", "sex-13",
    "sex-14", "sex-15", "sex-16", "sex-17", "sex-18", "sex-19", "sex-20",
    "sex-21", "sex-22", "sex-23", "sex-24", "sex-25", "sex-26", "sex-27",
    "sex-28", "sex-29", "sex-30", "sex-31", "sex-32", "sex-33", "sex-34",
    "sgr-00",
    "tri-00", "tri-01", "tri-02", #"tri-03",
    "uma-00", "uma-01", "uma-02", "uma-03", "uma-04", "uma-05", "uma-06",
    "uma-07", "uma-08", "uma-09", "uma-10", "uma-11",
    "vir-00", "vir-01", "vir-02", "vir-03", "vir-04", "vir-05", "vir-06",
    "vir-07", "vir-08", "vir-09", "vir-10", "vir-11", "vir-12", "vir-13",
    "vir-14", "vir-15", "vir-16",
]

filters = ['f115wn', 'f150wn', 'f200wn']

print(len(fields), "fields")

detection = Table.read(f"imaging/{fields[0]}/{fields[0]}-ir.cat.fits")
print(detection.colnames)
detection[:5]

In [ ]:
import requests
import concurrent.futures
from tqdm.auto import tqdm

def download_from_url(url, out_path, overwrite=False):
    """Download one url to out_path. Returns 'skipped' or 'downloaded'."""
    if os.path.exists(out_path) and not overwrite:
        return "skipped"
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    r = requests.get(url, auth=(os.environ["OUTTHERE_USER"], os.environ["OUTTHERE_PASSWORD"]))
    r.raise_for_status()
    with open(out_path, "wb") as f:
        f.write(r.content)
    return "downloaded"

def multi_thread_download(jobs, nworkers=20, overwrite=False, desc="download"):
    """jobs: list of (url, out_path). Returns dict with succeeded/skipped/failed stats."""
    stats = {"downloaded": [], "skipped": [], "failed": {}}
    with concurrent.futures.ThreadPoolExecutor(max_workers=nworkers) as executor:
        fut_to_job = {
            executor.submit(download_from_url, url, out_path, overwrite): (url, out_path)
            for url, out_path in jobs
        }
        for fut in tqdm(concurrent.futures.as_completed(fut_to_job),
                        total=len(fut_to_job), desc=desc):
            url, out_path = fut_to_job[fut]
            try:
                status = fut.result()      # 'skipped' or 'downloaded'
                stats[status].append(out_path)
            except Exception as e:
                stats["failed"][url] = repr(e)

    print(f"{desc}: {len(stats['downloaded'])} downloaded, "
          f"{len(stats['skipped'])} skipped, {len(stats['failed'])} failed")
    #if stats["failed"]:
        #for url, err in list(stats["failed"].items())[:10]:
            #print("  FAIL", url, "->", err)
    return stats

# download {field}-ir.cat.fits (detection catalog) for all fields
jobs = [
    (f"https://outthere-mpia.org/s3/data/{field}/{field}-ir.cat.fits",
     f"imaging/{field}/{field}-ir.cat.fits")
    for field in fields
]
stats = multi_thread_download(jobs, nworkers=20, desc="catalogs")

# download {field}-ir.cat.fits (detection catalog) for all fields
jobs = [
    (f"https://outthere-mpia.org/s3/data/{field}/{field}_fitresults.fits",
     f"imaging/{field}/{field}_fitresults.fits")
    for field in fields
]
stats = multi_thread_download(jobs, nworkers=20, desc="fitresults")

# download the drc_sci.fits for all fields and filters.
# not all fields have all filters, so some downloads will fail (counted in stats['failed']).
jobs = [
    (f"https://outthere-mpia.org/s3/data/{field}/{field}-{filt}-clear_drc_sci.fits",
     f"imaging/{field}/{field}-{filt}-clear_drc_sci.fits")
    for field in fields
    for filt in filters
]
stats = multi_thread_download(jobs, nworkers=20, desc="perfilter_sci")


#download the segmentation map for all fields.
jobs = [
    (f"https://outthere-mpia.org/s3/data/{field}/{field}-ir_seg.fits",
     f"imaging/{field}/{field}-ir_seg.fits")
    for field in fields
]
stats = multi_thread_download(jobs, nworkers=20, desc="segmentation maps")

In [ ]:
for field in fields[30:31]:
    #field = 'uma-03'
    for filt in filters:
        path = f"imaging/{field}/{field}-{filt}-clear_drc_sci.fits"
        if os.path.exists(path):
            with fits.open(path) as hdul:
                data = hdul[0].data
                #print units
                #comute pixel scale from CD1_1 and CD2_2
                cd1_1 = hdul[0].header.get("CD1_1", 0)
                cd2_2 = hdul[0].header.get("CD2_2", 0)
                pixel_scale_x = abs(cd1_1) * 3600 * 1000  # arcsec/pixel->mas
                pixel_scale_y = abs(cd2_2) * 3600 * 1000  # arcsec/pixel->mas
                pixel_scale = (pixel_scale_x, pixel_scale_y)
                print("Image data units:", hdul[0].header.get("BUNIT", "unknown"), "pixel scale:", pixel_scale)
                detection = Table.read(f"imaging/{field}/{field}-ir.cat.fits")
                fit_results = Table.read(f"imaging/{field}/{field}_fitresults.fits")
                #visualize one random image cutout with detections
                index = np.random.randint(len(fit_results))
                ra, dec = fit_results['ra'][index], fit_results['dec'][index]
                #convert from ra, dec to pixel coordinates
                wcs = WCS(hdul[0].header)
                x, y = wcs.world_to_pixel_values(ra, dec)
                x, y = int(x), int(y)
                size = 128
                cutout = data[y-size//2:y+size//2, x-size//2:x+size//2]
                #if empty pixels frac > 0.5
                empty_frac = np.mean(cutout == 0)
                print(f"Empty pixels fraction in cutout: {empty_frac:.2%}")

                vmin, vmax = np.percentile(cutout, [0.5, 99.5])
                plt.imshow(cutout, origin='lower', cmap='gray', vmin=vmin, vmax=vmax)
                plt.scatter(size//2, size//2, s=100, edgecolor='red', facecolor='none')
                plt.title(f"{field} {filt} cutout with detection, shape = {cutout.shape}")
                plt.show()
                
                #checkout the segmentation map for the same location
                seg_path = f"imaging/{field}/{field}-ir_seg.fits"
                if os.path.exists(seg_path):
                    with fits.open(seg_path) as seg_hdul:
                        seg_data = seg_hdul[0].data
                        seg_cutout = seg_data[y-size//2:y+size//2, x-size//2:x+size//2]
                        plt.imshow(seg_cutout, origin='lower', cmap='tab20')
                        plt.scatter(size//2, size//2, s=100, edgecolor='red', facecolor='none')
                        plt.title(f"{field} {filt} segmentation map cutout")
                        plt.show()
                spectra_fitting = Table.read(f"imaging/{field}/{field}_fitresults.fits")
                detection = Table.read(f"imaging/{field}/{field}-ir.cat.fits")
                
                

In [ ]:
num_fit_results = 0
num_detections = 0
for field in fields:
    try:
        spectra_fitting = Table.read(f"imaging/{field}/{field}_fitresults.fits")
        detection = Table.read(f"imaging/{field}/{field}-ir.cat.fits")
        num_fit_results += len(spectra_fitting)
        num_detections += len(detection)
        print(field)
        print(len(spectra_fitting), "fit results,", len(detection), "detections")
    except Exception as e:
        print(f"Error occurred while processing {field}: {e}")
print(f"Total fit results: {num_fit_results}, total detections: {num_detections}")
print(f'Total detections: {num_detections}, total fit results: {num_fit_results}, average fit results per detection: {num_fit_results/num_detections:.2f}')